# 🧠 Leakage-Free Ensemble Spiking Neural Network for Network Intrusion Detection

[![Python 3.12](https://img.shields.io/badge/Python-3.12-blue.svg)](https://www.python.org/downloads/)
[![PyTorch](https://img.shields.io/badge/PyTorch-2.x-ee4c2c.svg)](https://pytorch.org/)
[![snnTorch](https://img.shields.io/badge/snnTorch-0.9-green.svg)](https://snntorch.readthedocs.io/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](LICENSE)

**An academic-grade Intrusion Detection System (IDS) built on the CSE-CIC-IDS2018 dataset.**

## Key Highlights

| Feature | Detail |
|---|---|
| **Architecture** | Ensemble of Rate-Coded, Temporal-Coded, and Population-Coded SNNs with a learned meta-learner |
| **Data Protocol** | Chronological Stratified Split — fully **leakage-free** |
| **Class Balancing** | BorderlineSMOTE with dynamic class weights |
| **Loss Function** | Focal Loss (γ=2.0) with label smoothing |
| **Baselines** | ANN (feed-forward) and Random Forest for comparison |

---

## 📑 Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Configuration & Hyperparameters](#2-configuration--hyperparameters)
3. [Data Loading & Feature Engineering](#3-data-loading--feature-engineering)
4. [Model Architecture](#4-model-architecture)
   - 4.1 Rate-Coded SNN
   - 4.2 Temporal-Coded SNN
   - 4.3 Hybrid Population SNN
   - 4.4 Ensemble SNN (Meta-Learner)
   - 4.5 Baseline ANN
5. [Loss Function — Focal Loss](#5-loss-function--focal-loss)
6. [Visualization Utilities](#6-visualization-utilities)
7. [Training Pipeline](#7-training-pipeline)
8. [Execution — Main Experiment](#8-execution--main-experiment)
9. [Baseline Comparison](#9-baseline-comparison)
   - 9.1 Data Preparation
   - 9.2 Baseline ANN Training
   - 9.3 Baseline Random Forest Training

---

<a id='1-environment-setup'></a>
## 1. Environment Setup

Install the required packages. This notebook was developed and tested on **Kaggle** with a **Tesla T4 GPU**.


In [1]:
!pip install -q snntorch seaborn imbalanced-learn tqdm
print('✅ Gerekli kütüphaneler kuruldu.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.6/125.6 kB 7.0 MB/s eta 0:00:00
✅ Gerekli kütüphaneler kuruldu.


<a id='2-configuration--hyperparameters'></a>
## 2. Configuration & Hyperparameters

All hyperparameters and global constants are defined in one place for reproducibility.

| Parameter | Value | Description |
|---|---|---|
| `SEED` | 42 | Random seed for full reproducibility |
| `LR` | 4×10⁻⁴ | Learning rate (AdamW) |
| `WEIGHT_DECAY` | 1×10⁻³ | L2 regularization strength |
| `NUM_EPOCHS` | 35 | Maximum training epochs |
| `PATIENCE` | 20 | Early stopping patience |
| `GRAD_CLIP` | 0.5 | Gradient clipping max norm |
| `FOCAL_GAMMA` | 2.0 | Focal loss focusing parameter |
| `LABEL_SMOOTH` | 0.05 | Label smoothing factor |
| `BATCH_SIZE` | 1024 | Mini-batch size |


In [2]:
import torch
import random
import numpy as np

# ─── Seed & Device ────────────────────────────────────────────────────
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ─── Data & Paths ─────────────────────────────────────────────────────
DATA_PATH = '/kaggle/input/datasets/solarmainframe/ids-intrusion-csv'
TASK_TYPE = "multiclass"
MAX_PER_FILE      = 600_000
MAX_TRAIN_SAMPLES = 800_000
BATCH_SIZE        = 1024

LABEL_COL_CANDIDATES = ("Label", "label", "LABEL")
BENIGN_LABEL_ALIASES = ("benign", "normal")
FEATURE_CORR_THRESHOLD = 0.98
DROP_CLASSES = ()

# ─── Training & Loss ──────────────────────────────────────────────────
LR           = 4e-4             
WEIGHT_DECAY = 1e-3             
NUM_EPOCHS   = 35
PATIENCE     = 20
GRAD_CLIP    = 0.5              

CLASS_WEIGHT_CEILING = 15.0     
USE_FOCAL_LOSS = True           
FOCAL_GAMMA = 2.0               
LABEL_SMOOTH = 0.05

CALIBRATE_CLASS_MULTIPLIERS = True 
USE_CLASS_BALANCED_SAMPLER = False

<a id='3-data-loading--feature-engineering'></a>
## 3. Data Loading & Feature Engineering

This section implements the complete data pipeline:

1. **CSV Loading** — Reads multiple CSV files from the IDS2018 dataset with per-file capping
2. **Label Normalization** — Ensures `Benign` is always class 0; merges rare web attack sub-types
3. **Feature Selection** — Removes zero-variance features, near-constant columns (>98% single value), and highly correlated features (ρ > 0.98)
4. **Chronological Stratified Split** — Per-class temporal ordering to prevent **data leakage** (70/10/20 train/val/test)
5. **RobustScaler** — Fitted **only on training data**, then applied to validation and test sets
6. **BorderlineSMOTE** — Applied **only on training data** after scaling to handle class imbalance


In [3]:
import os
import gc
import glob
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, LabelEncoder
from imblearn.over_sampling import BorderlineSMOTE
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

class IDSDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

def _find_csv_files(path: str) -> list[str]:
    if os.path.isfile(path) and path.lower().endswith(".csv"): return [path]
    return sorted(glob.glob(os.path.join(path, "**", "*.csv"), recursive=True))

def _find_label_column(columns) -> str:
    cols = list(columns)
    for cand in LABEL_COL_CANDIDATES:
        if cand in cols: return cand
    low = {c.strip().lower(): c for c in cols}
    if "label" in low: return low["label"]
    return cols[-1]

def _normalize_labels(y_raw: pd.Series, task_type: str):
    y_str = y_raw.astype(str).str.strip()
    y_low = y_str.str.lower()
    benign_mask = y_low.isin(BENIGN_LABEL_ALIASES) | (y_low == "benign")

    le = LabelEncoder()
    y_enc = le.fit_transform(y_str)
    classes = le.classes_.astype(object)

    benign_candidates = [i for i, c in enumerate(classes) if str(c).strip().lower() in BENIGN_LABEL_ALIASES or str(c).strip().lower() == "benign"]
    if not benign_candidates: return y_enc, classes

    benign_idx = benign_candidates[0]
    if benign_idx == 0: return y_enc, classes

    y_new = y_enc.copy()
    y_new[y_enc == 0] = benign_idx
    y_new[y_enc == benign_idx] = 0
    classes_new = classes.copy()
    classes_new[0], classes_new[benign_idx] = classes_new[benign_idx], classes_new[0]
    return y_new, classes_new

def load_data(dataset_path):
    csv_files = _find_csv_files(dataset_path)
    print(f'\nLoading data from {dataset_path} ({len(csv_files)} CSV files)...')
    all_dfs = []
    for filepath in csv_files:
        chunks, total = [], 0
        for chunk in pd.read_csv(filepath, low_memory=False, chunksize=100_000):
            chunks.append(chunk)
            total += len(chunk)
            if total >= MAX_PER_FILE: break
        df = pd.concat(chunks, ignore_index=True)
        if len(df) > MAX_PER_FILE: df = df.sample(n=MAX_PER_FILE, random_state=SEED)
        
        label_col = _find_label_column(df.columns)
        first_feat = df.columns[0]
        df = df[df[label_col] != label_col]
        df = df[pd.to_numeric(df[first_feat], errors='coerce').notna()]
        all_dfs.append(df)
        del chunks; gc.collect()
    
    combined = pd.concat(all_dfs, ignore_index=True)
    del all_dfs; gc.collect()
    return combined

def feature_engineering(combined, task_type: str = "multiclass"):
    label_col = _find_label_column(combined.columns)
    for col in combined.columns:
        if col != label_col: combined[col] = pd.to_numeric(combined[col], errors='coerce')

    combined = combined.replace([np.inf, -np.inf], np.nan).fillna(0)
    combined[label_col] = combined[label_col].astype(str).str.strip()

    web_attacks = ["Brute Force -Web", "Brute Force -XSS", "SQL Injection"]
    combined[label_col] = combined[label_col].replace(web_attacks, "Web Attacks")

    y_raw = combined[label_col]
    X_raw = combined.drop(columns=[label_col])
    X_raw = X_raw.select_dtypes(include=[np.number])

    zero_var = X_raw.columns[X_raw.var() == 0].tolist()
    if zero_var: X_raw = X_raw.drop(columns=zero_var)

    low_info = [c for c in X_raw.columns if X_raw[c].value_counts(normalize=True).iloc[0] > 0.98]
    if low_info: X_raw = X_raw.drop(columns=low_info)

    sample = X_raw.sample(min(50_000, len(X_raw)), random_state=SEED)
    corr = sample.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    high_corr = [c for c in upper.columns if any(upper[c] > FEATURE_CORR_THRESHOLD)]
    if high_corr: X_raw = X_raw.drop(columns=high_corr)

    feature_names = X_raw.columns.tolist()
    y, classes = _normalize_labels(y_raw, task_type=task_type)
    le = LabelEncoder()
    le.classes_ = np.array(classes, dtype=object)

    return X_raw.values.astype(np.float32), y, feature_names, le

def split_and_scale(X_values, y):
    print('\n★ Applying Chronological Split (Leakage-Free Protocol)...')
    train_idx, val_idx, test_idx = [], [], []
    
    unique_classes = np.unique(y)
    for cls in unique_classes:
        cls_indices = np.where(y == cls)[0]
        n_samples = len(cls_indices)
        t_bound = int(n_samples * 0.70)
        v_bound = int(n_samples * 0.80)
        
        train_idx.extend(cls_indices[:t_bound])
        val_idx.extend(cls_indices[t_bound:v_bound])
        test_idx.extend(cls_indices[v_bound:])

    train_idx, val_idx, test_idx = np.array(train_idx), np.array(val_idx), np.array(test_idx)
    np.random.seed(SEED)
    np.random.shuffle(train_idx) 

    X_train, y_train = X_values[train_idx], y[train_idx]
    X_val, y_val     = X_values[val_idx], y[val_idx]
    X_test, y_test   = X_values[test_idx], y[test_idx]

    scaler = RobustScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)
    return X_train, X_val, X_test, y_train, y_val, y_test, scaler

def apply_smote_and_cap(X_train, y_train):
    print(f'\n★ Balancing data with BorderlineSMOTE (Optimized)...')
    
    # 1. HIZLANDIRMA: SMOTE'dan önce eğitim verisini 500.000'e kırpıyoruz (Leakage yapmaz)
    # Çünkü milyonlarca satırı SMOTE'a sokmak saatler sürer.
    if len(X_train) > 500_000:
        _, X_train, _, y_train = train_test_split(
            X_train, y_train, train_size=500_000, 
            random_state=SEED, stratify=y_train
        )
        print(f'  Pre-SMOTE cap to {len(X_train):,} samples to prevent CPU freeze.')

    pre_smote_counts = Counter(y_train)
    max_count = max(pre_smote_counts.values())
    print(f'  Before: {len(X_train):,} samples, {len(pre_smote_counts)} classes')
    
    smote_target = max(int(max_count * 0.05), 10_000)
    strategy = {cls: smote_target for cls, cnt in pre_smote_counts.items() if cnt < smote_target}

    if strategy:
        min_cnt = min(pre_smote_counts.values())
        k = min(5, max(min_cnt - 1, 1))
        try:
            smote = BorderlineSMOTE(random_state=SEED, k_neighbors=k, sampling_strategy=strategy)
            X_train, y_train = smote.fit_resample(X_train, y_train)
            print(f'  After BorderlineSMOTE: {len(X_train):,} samples')
        except Exception as e:
            print(f'  BorderlineSMOTE failed: {e} — using class weights only')

    # Eğitim seti hala çok büyükse nihai bir sınır koy (Güvenlik kemeri)
    if len(X_train) > MAX_TRAIN_SAMPLES:
        _, X_train, _, y_train = train_test_split(
            X_train, y_train, test_size=MAX_TRAIN_SAMPLES, 
            random_state=SEED, stratify=y_train
        )
        print(f'  Final Stratified cap to {len(X_train):,} samples')

    return X_train, y_train

def create_dataloaders(X_train, y_train, X_val, y_val, X_test, y_test):
    g = torch.Generator(); g.manual_seed(SEED)
    train_loader = DataLoader(IDSDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True, generator=g)
    val_loader = DataLoader(IDSDataset(X_val, y_val), batch_size=BATCH_SIZE * 2, shuffle=False)
    test_loader = DataLoader(IDSDataset(X_test, y_test), batch_size=BATCH_SIZE * 2, shuffle=False)
    return train_loader, val_loader, test_loader

<a id='4-model-architecture'></a>
## 4. Model Architecture

The core architecture is an **Ensemble of three Spiking Neural Networks**, each using a different neural coding scheme:

### 4.1 Rate-Coded SNN (`RateCodedSNN`)
- Encodes input features as **firing rates** via Bernoulli sampling
- 3-layer LIF (Leaky Integrate-and-Fire) network: 256 → 128 → 64
- Learnable membrane decay parameter (β ≈ 0.85)

### 4.2 Temporal-Coded SNN (`TemporalCodedSNN`)
- Encodes features as **spike timing** — higher values fire earlier
- GELU-activated encoder → 2-layer LIF network: 512 → 256 → 128
- Uses arctangent surrogate gradient (α=2.0)

### 4.3 Hybrid Population SNN (`HybridPopulationSNN`)
- **Population encoding** expands input dimensionality (×4)
- 3-layer LIF network with **residual connection**: 384 → 192 → 96
- Combines rate-coding with population diversity

### 4.4 Ensemble SNN (`EnsembleSNN`)
- **Weighted average** of three SNN outputs (learnable softmax weights)
- **Meta-learner** MLP that fuses concatenated outputs
- Final prediction: `0.6 × weighted_avg + 0.4 × meta_learner`

### 4.5 Baseline ANN (`ANNOnly`)
- Feed-forward network with residual connection: 512 → 512 → 256 → num_classes
- Used for comparison against the SNN ensemble

```
                    ┌───────────────────┐
                    │    Input (x)       │
                    └─────┬─────────────┘
              ┌───────────┼───────────────┐
              ▼           ▼               ▼
       ┌──────────┐ ┌──────────┐  ┌──────────────┐
       │ Rate SNN │ │Temporal  │  │  Hybrid Pop  │
       │(Bernoulli)│ │SNN(Time)│  │  SNN (×4)    │
       └────┬─────┘ └────┬─────┘  └──────┬───────┘
            │             │               │
            ▼             ▼               ▼
       ┌────────────────────────────────────────┐
       │  Learnable Softmax Weights (w₁,w₂,w₃) │
       │  + Meta-Learner MLP (concat → 64 → C) │
       └────────────────┬───────────────────────┘
                        ▼
              ┌───────────────────┐
              │  Final Prediction │
              │  0.6·W + 0.4·Meta │
              └───────────────────┘
```


In [4]:
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate

class RateEncoder(nn.Module):
    def __init__(self, num_steps: int):
        super().__init__()
        self.num_steps = num_steps
    def forward(self, x):
        x_prob = torch.sigmoid(x)
        spikes = [torch.bernoulli(x_prob) for _ in range(self.num_steps)]
        return torch.stack(spikes, dim=0) 

class RateCodedSNN(nn.Module):
    def __init__(self, input_size, num_classes, hidden_dims=[256, 128, 64], beta=0.85, num_steps=12, dropout=0.3):
        super().__init__()
        self.num_steps = num_steps
        spike_grad = surrogate.fast_sigmoid(slope=25)
        layers = []
        prev_dim = input_size
        for h_dim in hidden_dims:
            layers.extend([nn.Linear(prev_dim, h_dim), nn.BatchNorm1d(h_dim), snn.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=True, learn_beta=True), nn.Dropout(dropout)])
            prev_dim = h_dim
        self.features = nn.Sequential(*layers)
        self.output_fc = nn.Linear(prev_dim, num_classes)
        self.output_lif = snn.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=True, learn_beta=True, output=True)
    
    def forward(self, x):
        x_rate = torch.sigmoid(x)
        mem_rec = []
        for layer in self.features:
            if hasattr(layer, 'reset_hidden'): layer.reset_hidden()
        self.output_lif.reset_hidden()
        
        for step in range(self.num_steps):
            cur = (torch.rand_like(x_rate) < x_rate).float()
            for layer in self.features: cur = layer(cur)
            spk, mem = self.output_lif(self.output_fc(cur))
            mem_rec.append(mem)
        return torch.stack(mem_rec).mean(dim=0)

class TemporalCodedSNN(nn.Module):
    def __init__(self, input_size, num_classes, hidden_dims=[512, 256, 128], beta=0.90, num_steps=12, dropout=0.3):
        super().__init__()
        self.num_steps = num_steps
        spike_grad = surrogate.atan(alpha=2.0)
        self.encoder = nn.Sequential(nn.Linear(input_size, hidden_dims[0]), nn.BatchNorm1d(hidden_dims[0]), nn.GELU(), nn.Dropout(dropout * 0.5))
        self.fc1 = nn.Linear(hidden_dims[0], hidden_dims[1])
        self.bn1 = nn.BatchNorm1d(hidden_dims[1])
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=True, learn_beta=True)
        self.fc2 = nn.Linear(hidden_dims[1], hidden_dims[2])
        self.bn2 = nn.BatchNorm1d(hidden_dims[2])
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=True, learn_beta=True)
        self.fc_out = nn.Linear(hidden_dims[2], num_classes)
        self.lif_out = snn.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=True, learn_beta=True, output=True)
        self.drop = nn.Dropout(dropout)
    
    def forward(self, x):
        encoded = self.encoder(x)
        spike_times = ((1.0 - torch.sigmoid(encoded)) * (self.num_steps - 1)).long().clamp(0, self.num_steps - 1)
        mem_rec = []
        self.lif1.reset_hidden(); self.lif2.reset_hidden(); self.lif_out.reset_hidden()
        for step in range(self.num_steps):
            spike_input = (spike_times == step).float()
            spk1 = self.drop(self.lif1(self.bn1(self.fc1(spike_input))))
            spk2 = self.drop(self.lif2(self.bn2(self.fc2(spk1))))
            _, mem_out = self.lif_out(self.fc_out(spk2))
            mem_rec.append(mem_out)
        return torch.stack(mem_rec).mean(dim=0)

class HybridPopulationSNN(nn.Module):
    def __init__(self, input_size, num_classes, hidden_dims=[384, 192, 96], beta=0.88, num_steps=12, dropout=0.3, num_populations=4):
        super().__init__()
        self.num_steps = num_steps
        spike_grad = surrogate.fast_sigmoid(slope=20)
        pop_size = input_size * num_populations
        self.population_encoder = nn.Sequential(nn.Linear(input_size, pop_size), nn.BatchNorm1d(pop_size), nn.ReLU())
        self.fc1 = nn.Linear(pop_size, hidden_dims[0]); self.bn1 = nn.BatchNorm1d(hidden_dims[0]); self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=True, learn_beta=True)
        self.fc2 = nn.Linear(hidden_dims[0], hidden_dims[1]); self.bn2 = nn.BatchNorm1d(hidden_dims[1]); self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=True, learn_beta=True)
        self.res_proj = nn.Linear(hidden_dims[0], hidden_dims[1])
        self.fc3 = nn.Linear(hidden_dims[1], hidden_dims[2]); self.bn3 = nn.BatchNorm1d(hidden_dims[2]); self.lif3 = snn.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=True, learn_beta=True)
        self.fc_out = nn.Linear(hidden_dims[2], num_classes); self.lif_out = snn.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=True, learn_beta=True, output=True)
        self.drop = nn.Dropout(dropout)
    
    def forward(self, x):
        pop_rate = torch.sigmoid(self.population_encoder(x))
        mem_rec = []
        self.lif1.reset_hidden(); self.lif2.reset_hidden(); self.lif3.reset_hidden(); self.lif_out.reset_hidden()
        for step in range(self.num_steps):
            spike_input = (torch.rand_like(pop_rate) < pop_rate).float()
            spk1 = self.drop(self.lif1(self.bn1(self.fc1(spike_input))))
            spk2 = self.drop(self.lif2(self.bn2(self.fc2(spk1))) + self.res_proj(spk1))
            spk3 = self.drop(self.lif3(self.bn3(self.fc3(spk2))))
            _, mem_out = self.lif_out(self.fc_out(spk3))
            mem_rec.append(mem_out)
        return torch.stack(mem_rec).mean(dim=0)

class EnsembleSNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.model_rate = RateCodedSNN(input_size, num_classes)
        self.model_temporal = TemporalCodedSNN(input_size, num_classes)
        self.model_hybrid = HybridPopulationSNN(input_size, num_classes)
        self.ensemble_weights = nn.Parameter(torch.ones(3) / 3.0)
        self.meta_learner = nn.Sequential(nn.Linear(num_classes * 3, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, num_classes))
    
    def forward(self, x):
        out_rate, out_temporal, out_hybrid = self.model_rate(x), self.model_temporal(x), self.model_hybrid(x)
        w = torch.softmax(self.ensemble_weights, dim=0)
        weighted = w[0] * out_rate + w[1] * out_temporal + w[2] * out_hybrid
        meta_out = self.meta_learner(torch.cat([out_rate, out_temporal, out_hybrid], dim=1))
        return 0.6 * weighted + 0.4 * meta_out

    def forward_debug(self, x):
        out = self.forward(x)
        return {'output': out, 'spk1': torch.zeros((10, x.size(0), 10)).to(x.device), 'mem_out': out.unsqueeze(0).repeat(10, 1, 1)}

class ANNOnly(nn.Module):
    def __init__(self, inp_size, n_classes, hidden=512, drop=0.3):
        super().__init__()
        self.layer1 = nn.Sequential(nn.Linear(inp_size, hidden), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(drop))
        self.layer2 = nn.Sequential(nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(drop))
        self.head = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.GELU(), nn.Dropout(drop), nn.Linear(hidden // 2, n_classes))
    def forward(self, x):
        h1 = self.layer1(x)
        return self.head(self.layer2(h1) + h1)

<a id='5-loss-function--focal-loss'></a>
## 5. Loss Function — Focal Loss

We use **Focal Loss** ([Lin et al., 2017](https://arxiv.org/abs/1708.02002)) to address the severe class imbalance in network traffic data.

$$\mathcal{L}_{\text{focal}} = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

- **γ = 2.0** — Down-weights easy (well-classified) examples
- **Label smoothing = 0.05** — Prevents overconfident predictions
- **Dynamic class weights** — Inversely proportional to class frequency, capped at 15×


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, class_weights=None, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.register_buffer('class_weights', class_weights if class_weights is not None else torch.tensor(1.0))

    def forward(self, logits, targets):
        num_classes = logits.size(1)
        with torch.no_grad():
            smooth = self.label_smoothing / num_classes
            one_hot = torch.zeros_like(logits).scatter_(1, targets.unsqueeze(1), 1.0)
            one_hot = one_hot * (1.0 - self.label_smoothing) + smooth

        log_probs = F.log_softmax(logits, dim=1)
        probs = log_probs.exp()
        focal_weight = (1.0 - probs).pow(self.gamma)
        loss = -(focal_weight * one_hot * log_probs).sum(dim=1)

        if self.class_weights.dim() > 0:
            loss = loss * self.class_weights[targets]
        return loss.mean()

<a id='6-visualization-utilities'></a>
## 6. Visualization Utilities

Helper functions for generating publication-quality plots:
- **Training History** — Loss, accuracy, F1 scores, ADR, error rates, and learning rate curves
- **Confusion Matrix** — Normalized and raw count versions (side-by-side)
- **Per-Class F1 Scores** — Color-coded bar chart (green > 0.9, yellow > 0.7, red otherwise)
- **ROC Curves** — One-vs-rest ROC with per-class and macro-average AUC


In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize

def plot_training_history(history, num_epochs, out_path='training_history.png'):
    fig, axes = plt.subplots(2, 3, figsize=(20, 11))
    fig.suptitle('IDS2018 Ensemble SNN — Training History', fontsize=16, fontweight='bold')
    epochs_range = range(1, len(history['train_loss']) + 1)
    
    axes[0, 0].plot(epochs_range, history['train_loss'], 'b-', label='Train'); axes[0, 0].plot(epochs_range, history['val_loss'], 'r-', label='Val')
    axes[0, 0].set_title('Loss'); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].plot(epochs_range, history['train_acc'], 'b-', label='Train'); axes[0, 1].plot(epochs_range, history['val_acc'], 'r-', label='Val')
    axes[0, 1].set_title('Accuracy'); axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)
    
    axes[0, 2].plot(epochs_range, history['val_f1'], 'g-', label='F1 weighted'); axes[0, 2].plot(epochs_range, history['val_f1_macro'], color='orange', label='F1 macro')
    axes[0, 2].set_title('Validation F1'); axes[0, 2].legend(); axes[0, 2].grid(True, alpha=0.3)
    
    axes[1, 0].plot(epochs_range, history['val_adr'], 'darkgreen', lw=2); axes[1, 0].set_title('Attack Detection Rate'); axes[1, 0].grid(True, alpha=0.3)
    axes[1, 1].plot(epochs_range, history['val_fpr'], 'orange', label='FPR'); axes[1, 1].plot(epochs_range, history['val_fnr'], 'red', label='FNR')
    axes[1, 1].set_title('Error Rates'); axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)
    axes[1, 2].plot(epochs_range, history['lr'], 'purple', lw=2); axes[1, 2].set_title('Learning Rate'); axes[1, 2].set_yscale('log'); axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout(); plt.savefig(out_path, dpi=150, bbox_inches='tight'); plt.close()

def plot_confusion_matrix(tt, tp, unique_labels, used_names, out_path='confusion_matrix.png'):
    cm = confusion_matrix(tt, tp, labels=unique_labels)
    cm_norm = np.nan_to_num(cm.astype('float') / cm.sum(axis=1, keepdims=True))
    fig, axes = plt.subplots(1, 2, figsize=(24, 10))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[0], xticklabels=used_names, yticklabels=used_names, vmin=0, vmax=1)
    axes[0].set_title('Confusion Matrix (Normalized)', fontweight='bold')
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1], xticklabels=used_names, yticklabels=used_names)
    axes[1].set_title('Confusion Matrix (Counts)', fontweight='bold')
    for ax in axes: ax.tick_params(axis='x', rotation=45)
    plt.tight_layout(); plt.savefig(out_path, dpi=150, bbox_inches='tight'); plt.close()

def plot_f1_per_class(f1_per_class, used_names, out_path='f1_per_class.png'):
    fig, ax = plt.subplots(figsize=(14, 7))
    colors = ['#27ae60' if s > 0.9 else '#f39c12' if s > 0.7 else '#e74c3c' for s in f1_per_class]
    bars = ax.bar(range(len(f1_per_class)), f1_per_class, color=colors, edgecolor='black')
    for bar, s in zip(bars, f1_per_class): ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.01, f'{s:.3f}', ha='center', fontweight='bold')
    ax.set_xticks(range(len(used_names))); ax.set_xticklabels(used_names, rotation=45, ha='right')
    ax.set_title('Per-Class F1 Scores', fontweight='bold')
    plt.tight_layout(); plt.savefig(out_path, dpi=150, bbox_inches='tight'); plt.close()

def plot_roc_curves(y_true, y_proba, class_names, out_path='roc_curves.png'):
    n_classes = y_proba.shape[1]
    y_bin = label_binarize(y_true, classes=range(n_classes))
    fig, ax = plt.subplots(figsize=(12, 9))
    colors = plt.cm.tab20(np.linspace(0, 1, n_classes))
    all_fpr, all_tpr = {}, {}
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
        all_fpr[i], all_tpr[i] = fpr, tpr
        ax.plot(fpr, tpr, color=colors[i], label=f'{class_names[i]} (AUC={auc(fpr, tpr):.3f})')
    
    all_fpr_grid = np.linspace(0, 1, 200)
    mean_tpr = np.zeros_like(all_fpr_grid)
    for i in range(n_classes): mean_tpr += np.interp(all_fpr_grid, all_fpr[i], all_tpr[i])
    ax.plot(all_fpr_grid, mean_tpr / n_classes, 'k--', lw=2.5, label=f'Macro avg (AUC={auc(all_fpr_grid, mean_tpr / n_classes):.3f})')
    ax.legend(loc='lower right', fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(out_path, dpi=150, bbox_inches='tight'); plt.close()

<a id='7-training-pipeline'></a>
## 7. Training Pipeline

The `main()` function orchestrates the entire experiment:

1. **Data preparation** — Load → Engineer → Split → Scale → SMOTE
2. **Model creation** — Instantiate `EnsembleSNN` on GPU
3. **Training loop** — AdamW optimizer with OneCycleLR scheduler
4. **Validation** — Per-epoch F1 (weighted + macro) tracking with early stopping
5. **Post-hoc calibration** — Optional class-multiplier optimization on validation set
6. **Test evaluation** — Classification report + all visualization plots


In [7]:
import time
import torch.nn.functional as F
from sklearn.metrics import classification_report, f1_score, recall_score, accuracy_score
from tqdm.auto import tqdm

def _optimize_class_multipliers(val_proba, val_targets, num_classes):
    multipliers = np.ones(num_classes, dtype=np.float32)
    best_score = f1_score(val_targets, val_proba.argmax(axis=1), average='macro', zero_division=0)
    search_grid = [1.0, 1.2, 1.5, 2.0]
    for cls in range(1, num_classes):
        cls_best_mul, cls_best_score = multipliers[cls], best_score
        for mul in search_grid:
            test_mul = multipliers.copy()
            test_mul[cls] = mul
            score = f1_score(val_targets, (val_proba * test_mul[None, :]).argmax(axis=1), average='macro', zero_division=0)
            if score > cls_best_score: cls_best_score, cls_best_mul = score, mul
        multipliers[cls], best_score = cls_best_mul, cls_best_score
    return multipliers, best_score

def main(dataset_path, task_type):
    set_seed()
    print('=' * 60 + '\n  IDS2018 ENSEMBLE SNN — ACADEMIC PIPELINE\n' + '=' * 60)

    combined_df = load_data(dataset_path)
    X_values, y, feature_names, le = feature_engineering(combined_df, task_type=task_type)
    input_size, num_classes = X_values.shape[1], len(set(y))

    X_train, X_val, X_test, y_train, y_val, y_test, scaler = split_and_scale(X_values, y)
    X_train, y_train = apply_smote_and_cap(X_train, y_train)
    train_loader, val_loader, test_loader = create_dataloaders(X_train, y_train, X_val, y_val, X_test, y_test)

    # MODEL OLUŞTURMA
    model = EnsembleSNN(input_size, num_classes).to(DEVICE)
    print(f'\n✅ Ensemble SNN created | Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

    # LOSS & OPTIMIZER
    counts = Counter(y_train)
    weights_arr = np.clip(np.array([len(y_train) / (num_classes * counts.get(i, 1)) for i in range(num_classes)]), 0.5, CLASS_WEIGHT_CEILING)
    class_weights = torch.FloatTensor(weights_arr / weights_arr.sum() * num_classes).to(DEVICE)
    
    criterion = FocalLoss(class_weights=class_weights, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTH)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, steps_per_epoch=len(train_loader), epochs=NUM_EPOCHS, pct_start=0.1)

    best_score, best_epoch, patience_cnt, best_state = 0, 0, 0, None
    history = {k: [] for k in ['train_loss', 'train_acc', 'val_loss', 'val_acc', 'val_f1', 'val_f1_macro', 'val_adr', 'val_fpr', 'val_fnr', 'lr']}

    print(f'\n🚀 TRAINING STARTED | Epochs: {NUM_EPOCHS}')
    train_start = time.time()

    for epoch in range(NUM_EPOCHS):
        ep_start = time.time()
        model.train()
        ep_loss, ep_correct, ep_total = 0, 0, 0

        pbar = tqdm(train_loader, desc=f'Ep {epoch+1}/{NUM_EPOCHS}', leave=False)
        for X_b, y_b in pbar:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad()
            out = model(X_b)
            loss = criterion(out, y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            ep_loss += loss.item() * X_b.size(0)
            ep_correct += (out.argmax(1) == y_b).sum().item()
            ep_total += y_b.size(0)
            pbar.set_postfix(loss=f'{loss.item():.4f}')

        # VALIDATION
        model.eval()
        v_loss, all_vp, all_vt, all_vproba = 0, [], [], []
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
                out = model(X_b)
                v_loss += criterion(out, y_b).item() * X_b.size(0)
                all_vp.extend(out.argmax(1).cpu().numpy())
                all_vt.extend(y_b.cpu().numpy())
                all_vproba.append(torch.softmax(out, dim=1).cpu().numpy())

        vp, vt = np.array(all_vp), np.array(all_vt)
        vproba = np.concatenate(all_vproba, axis=0)
        
        if CALIBRATE_CLASS_MULTIPLIERS and epoch >= 5:
            cmul, _ = _optimize_class_multipliers(vproba, vt, num_classes)
            vp = (vproba * cmul[None, :]).argmax(axis=1)

        val_f1, val_f1m = f1_score(vt, vp, average='weighted', zero_division=0), f1_score(vt, vp, average='macro', zero_division=0)
        attack_mask, benign_mask = vt > 0, vt == 0
        
        history['train_loss'].append(ep_loss / ep_total); history['train_acc'].append(ep_correct / ep_total)
        history['val_loss'].append(v_loss / len(vt)); history['val_acc'].append(accuracy_score(vt, vp))
        history['val_f1'].append(val_f1); history['val_f1_macro'].append(val_f1m)
        history['val_adr'].append((recall_score(vt[attack_mask] > 0, vp[attack_mask] > 0, average='binary', zero_division=0) if attack_mask.sum() > 0 else 0.0))
        history['val_fpr'].append(float((vp[benign_mask] > 0).mean()) if benign_mask.sum() > 0 else 0.0)
        history['val_fnr'].append(float((vp[attack_mask] == 0).mean()) if attack_mask.sum() > 0 else 0.0)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        val_score = 0.5 * val_f1 + 0.5 * val_f1m
        print(f"{'★' if val_score > best_score else ' '} Ep {epoch+1:2d} | L:{history['train_loss'][-1]:.3f}/{history['val_loss'][-1]:.3f} | F1w:{val_f1:.4f} F1m:{val_f1m:.4f} | {time.time()-ep_start:.0f}s")

        if val_score > best_score:
            best_score, best_epoch, patience_cnt = val_score, epoch + 1, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE: print(f'\n⏹ Early stopping'); break

    if best_state: model.load_state_dict(best_state)

    # TEST
    print(f'\n★ Test Evaluation...')
    model.eval()
    all_logits = []
    with torch.no_grad():
        for X_b, _ in test_loader: all_logits.append(model(X_b.to(DEVICE)).cpu())
    
    proba = torch.softmax(torch.cat(all_logits, dim=0), dim=1).numpy()
    class_multipliers = np.ones(num_classes, dtype=np.float32)
    
    if CALIBRATE_CLASS_MULTIPLIERS:
        all_v_logits, all_v_targets = [], []
        with torch.no_grad():
            for X_b, y_b in val_loader: all_v_logits.append(model(X_b.to(DEVICE)).cpu()); all_v_targets.extend(y_b.numpy())
        class_multipliers, _ = _optimize_class_multipliers(torch.softmax(torch.cat(all_v_logits, dim=0), dim=1).numpy(), np.array(all_v_targets), num_classes)
    
    tp = (proba * class_multipliers[None, :]).argmax(axis=1)
    used_names = [le.classes_[i] if i < len(le.classes_) else f'Class_{i}' for i in sorted(set(y_test) | set(tp))]

    print(f"\n{classification_report(y_test, tp, target_names=used_names, zero_division=0)}")
    
    # PLOTS & SAVE
    plot_training_history(history, NUM_EPOCHS)
    plot_confusion_matrix(y_test, tp, sorted(set(y_test) | set(tp)), used_names)
    plot_f1_per_class(f1_score(y_test, tp, average=None, zero_division=0), used_names)
    plot_roc_curves(y_test, proba, used_names)
    torch.save({'model_state_dict': model.state_dict()}, 'ensemble_snn.pth')
    print('✅ Saved ensemble_snn.pth and all plots.')

<a id='8-execution--main-experiment'></a>
## 8. Execution — Main Experiment

Run the full Ensemble SNN training and evaluation pipeline.


In [8]:
main(DATA_PATH, task_type=TASK_TYPE)

  IDS2018 ENSEMBLE SNN — ACADEMIC PIPELINE

Loading data from /kaggle/input/datasets/solarmainframe/ids-intrusion-csv (10 CSV files)...

★ Applying Chronological Split (Leakage-Free Protocol)...

★ Balancing data with BorderlineSMOTE (Optimized)...
  Pre-SMOTE cap to 3,091,742 samples to prevent CPU freeze.
  Before: 3,091,742 samples, 12 classes
  BorderlineSMOTE failed: BorderlineSMOTE.__init__() got an unexpected keyword argument 'n_jobs' — using class weights only
  Final Stratified cap to 800,000 samples

✅ Ensemble SNN created | Params: 501,270

🚀 TRAINING STARTED | Epochs: 35


Ep 1/35:   0%|          | 0/782 [00:00<?, ?it/s]

★ Ep  1 | L:0.292/0.367 | F1w:0.2610 F1m:0.0579 | 130s


Ep 2/35:   0%|          | 0/782 [00:00<?, ?it/s]

★ Ep  2 | L:0.115/0.137 | F1w:0.5567 F1m:0.3684 | 127s


...

  Ep 25 | L:0.061/0.111 | F1w:0.6881 F1m:0.5732 | 129s


Ep 26/35:   0%|          | 0/782 [00:00<?, ?it/s]

  Ep 26 | L:0.061/0.114 | F1w:0.7003 F1m:0.5713 | 129s


Ep 27/35:   0%|          | 0/782 [00:00<?, ?it/s]

  Ep 27 | L:0.061/0.117 | F1w:0.6665 F1m:0.5657 | 128s

⏹ Early stopping

★ Test Evaluation...

                          precision    recall  f1-score   support

                  Benign       0.95      0.95      0.95    687465
                     Bot       0.79      0.84      0.82     57239
        DDOS attack-HOIC       0.93      0.84      0.88     92339
    DDOS attack-LOIC-UDP       1.00      0.95      0.97       346
   DoS attacks-GoldenEye       0.80      0.97      0.88      8302
        DoS attacks-Hulk       0.97      0.97      0.97     52775
DoS attacks-SlowHTTPTest       0.27      0.75      0.40     18287
   DoS attacks-Slowloris       0.54      0.91      0.68      2198
          FTP-BruteForce       0.42      0.24      0.31     38672
           Infilteration       0.46      0.31      0.37     30890
          SSH-Bruteforce       0.95      0.91      0.93     37518
             Web Attacks       0.00      0.00      0.00       186

                accuracy                    

<a id='9-baseline-comparison'></a>
## 9. Baseline Comparison

To validate the effectiveness of the SNN ensemble, we compare against two standard baselines using **identical data splits, features, and preprocessing**.

### 9.1 Data Preparation

Re-initialize the data pipeline with the same seed and hyperparameters for a fair comparison.


In [ ]:
# --- Baseline Karsılastırmasi icin Veri Hazırlama ---
# SNN pipeline ile birebir aynı fonksiyonlar ve hiperparametreler kullanılır.
set_seed()
print('=' * 60)
print('  BASELINE COMPARISON — ANN & RANDOM FOREST')
print('=' * 60)

_combined_df = load_data(DATA_PATH)
_X_values, _y, _feature_names, _le = feature_engineering(_combined_df, task_type=TASK_TYPE)
_X_train, _X_val, _X_test, _y_train, _y_val, _y_test, _scaler = split_and_scale(_X_values, _y)
_X_train, _y_train = apply_smote_and_cap(_X_train, _y_train)
_train_loader, _val_loader, _test_loader = create_dataloaders(_X_train, _y_train, _X_val, _y_val, _X_test, _y_test)

_input_size = _X_train.shape[1]
_num_classes = len(_le.classes_)
print(f'\n✅ Baseline data ready | Input: {_input_size} | Classes: {_num_classes}')


### 9.2 Baseline ANN Training

A standard feed-forward neural network (2-layer MLP with residual connection) trained with the same optimizer, scheduler, and loss function as the SNN ensemble.


In [ ]:
# --- Baseline ANN Egitimi ---
# Mimari: g-version-6'daki ANNOnly sinifi kullanilir (Cell 4'te tanimli).
# Hiperparametreler: SNN ile tamamen ayni (LR, WEIGHT_DECAY, NUM_EPOCHS, PATIENCE, GRAD_CLIP).
import numpy as _np
from collections import Counter as _Counter

ann_model = ANNOnly(_input_size, _num_classes).to(DEVICE)
print(f'✅ Baseline ANN created | Params: {sum(p.numel() for p in ann_model.parameters() if p.requires_grad):,}')

_counts = _Counter(_y_train)
_w_arr = _np.clip(
    [len(_y_train) / (_num_classes * _counts.get(i, 1)) for i in range(_num_classes)],
    0.5, CLASS_WEIGHT_CEILING
)
_cw = torch.FloatTensor(_w_arr / _w_arr.sum() * _num_classes).to(DEVICE)

ann_criterion = FocalLoss(class_weights=_cw, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTH)
ann_optimizer = torch.optim.AdamW(ann_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
ann_scheduler = torch.optim.lr_scheduler.OneCycleLR(
    ann_optimizer, max_lr=LR,
    steps_per_epoch=len(_train_loader), epochs=NUM_EPOCHS, pct_start=0.1
)

ann_best_score, ann_best_state, ann_patience_cnt = 0, None, 0

for epoch in range(NUM_EPOCHS):
    ann_model.train()
    for X_b, y_b in _train_loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        ann_optimizer.zero_grad()
        loss = ann_criterion(ann_model(X_b), y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ann_model.parameters(), GRAD_CLIP)
        ann_optimizer.step()
        ann_scheduler.step()

    ann_model.eval()
    vp, vt = [], []
    with torch.no_grad():
        for X_b, y_b in _val_loader:
            vp.extend(ann_model(X_b.to(DEVICE)).argmax(1).cpu().numpy())
            vt.extend(y_b.numpy())
    vp, vt = _np.array(vp), _np.array(vt)
    f1w = f1_score(vt, vp, average='weighted', zero_division=0)
    f1m = f1_score(vt, vp, average='macro', zero_division=0)
    score = 0.5 * f1w + 0.5 * f1m
    marker = '★' if score > ann_best_score else ' '
    print(f'{marker} ANN Ep {epoch+1:2d} | F1w: {f1w:.4f} | F1m: {f1m:.4f}')
    if score > ann_best_score:
        ann_best_score, ann_patience_cnt = score, 0
        ann_best_state = {k: v.clone() for k, v in ann_model.state_dict().items()}
    else:
        ann_patience_cnt += 1
        if ann_patience_cnt >= PATIENCE:
            print('\n⏹ ANN Early Stopping')
            break

if ann_best_state:
    ann_model.load_state_dict(ann_best_state)

ann_model.eval()
ann_tp, ann_tt = [], []
with torch.no_grad():
    for X_b, y_b in _test_loader:
        ann_tp.extend(ann_model(X_b.to(DEVICE)).argmax(1).cpu().numpy())
        ann_tt.extend(y_b.numpy())

_used_ann = [_le.classes_[i] for i in sorted(set(ann_tt) | set(ann_tp)) if i < len(_le.classes_)]
print('\n✅ Baseline ANN — Test Results:')
print(classification_report(ann_tt, ann_tp, target_names=_used_ann, zero_division=0))


### 9.3 Baseline Random Forest Training

A scikit-learn `RandomForestClassifier` (100 trees, max depth 25) as a traditional ML baseline.


In [ ]:
# --- Baseline Random Forest Egitimi ---
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=25,
    random_state=SEED
)
rf_model.fit(_X_train, _y_train)

rf_preds = rf_model.predict(_X_test)
_used_rf = [_le.classes_[i] for i in sorted(set(_y_test) | set(rf_preds)) if i < len(_le.classes_)]
print('✅ Baseline Random Forest — Test Results:')
print(classification_report(_y_test, rf_preds, target_names=_used_rf, zero_division=0))


---

## 📄 Citation

If you use this code in your research, please cite:

```bibtex
@misc{ids2018_ensemble_snn,
  title   = {Leakage-Free Ensemble Spiking Neural Network for Network Intrusion Detection},
  author  = {Özgür Uçar},
  year    = {2026},
  url     = {https://github.com/ozgurucar/ids2018-ensemble-snn}
}
```

## 📝 License

This project is licensed under the MIT License — see the [LICENSE](LICENSE) file for details.